**_**CREATING** THE DATA MODEL STAR SCHEMA_**

In [0]:
class Gold_dimesions:
  def __init__(self,spark):
    self.spark = spark
  def new_dimension(self):
    new_query = """
    SELECT
      ROW_NUMBER() OVER(ORDER BY c.coustmer_id) AS coustomer_sk,
      c.coustmer_id,
      c.coustmer_key,
      c.coustmer_firstname,
      c.coustmer_lastname,
      c.marriage_status,
      CASE
          WHEN c.coustmer_gender <> 'n/a' THEN c.coustmer_gender
          ELSE COALESCE(a.gender,'n/a')
      END AS gender,
      c.coustmer_join_date,
      a.birth_date,
      z.CNTRY AS country
    FROM 
      lakehouse.sliver.cleaned_coustmer c
    LEFT JOIN 
      lakehouse.sliver.cust_az a
    ON
      c.coustmer_key = a.customer_number
    LEFT JOIN
      lakehouse.sliver.coustmer_az z
    ON
      c.coustmer_key = z.cid
    """
    #runing the sql query
    print("reading the data")
    new_data = self.spark.sql(new_query)
    #sending the data to the gold
    new_data.write.format("delta").mode("append").saveAsTable("lakehouse.gold.coust_dimension")
    print("sending the data to the gold layer coust_dimension")

In [0]:
gold_dim = Gold_dimesions(spark)
gold_dim.new_dimension()

**_CREATING THE DIMENSION PRODUCT
 _**

In [0]:
class dim_product:
    def __init__(self,spark):
        self.spark = spark
    def product_dim(self):
        sec_query = """
                select
                    ROW_NUMBER() OVER(ORDER BY p.product_id) as product_key,
                    p.product_id,
                    p.category_id,
                    p.product_number,
                    p.product_name,
                    p.product_line,
                    p.start_date,
                    x.category,
                    x.subcategory,
                    x.maintenance_flag
                from 
                    lakehouse.sliver.crm_product p
                left join 
                    lakehouse.sliver.cat_px x
                on
                    p.category_id = x.category_id """
        print("reading the data for dim_product")
        new_datas = self.spark.sql(sec_query)
        print("sending the data into the gold layer")
        new_datas.write.format("delta").mode("append").saveAsTable("lakehouse.gold.dim_product")
        

In [0]:
dim = dim_product(spark)
dim.product_dim()

**_FACT TABLE_**

In [0]:
class fact_dimension:
    def __init__(self, spark):
        self.spark = spark

    def dimension_fact(self):

        fact_dim = """
        SELECT
            s.order_number,
            g.coustomer_sk,
            p.product_key,
            s.order_date,
            s.ship_date,
            s.due_date,
            s.sales_amount,
            s.quantity,
            s.price
        FROM lakehouse.sliver.sales_details s
        LEFT JOIN lakehouse.gold.coust_dimension g
        ON s.customer_id = g.coustmer_id
        LEFT JOIN lakehouse.gold.dim_product p
        ON s.product_number = p.product_number
        """

        df = self.spark.sql(fact_dim)

        print("Sending data to Gold layer")

        df.write.format("delta").mode("overwrite").saveAsTable("lakehouse.gold.fact_table")


new_obj = fact_dimension(spark)
new_obj.dimension_fact()